# 108 — Guardrails AI: Output Schema Enforcement
## Validate, Repair, and Retry LLM Outputs Automatically
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/108-guardrails-ai/guardrails_ai_workbook.ipynb)

LLMs are probabilistic — they **sometimes** return valid JSON, **sometimes** forget a field, **sometimes** include a URL when you said not to. In production, you cannot rely on prompt engineering alone. You need a validation layer that can:

1. **Parse** the model output into a typed schema  
2. **Detect** every constraint violation  
3. **Reask** the model with precise correction instructions — automatically  
4. **Audit** every attempt with a full history trail  

**Guardrails AI** is the open-source framework that does exactly that. It wraps any LLM call with a `Guard` object backed by Pydantic v2 schemas and a reask loop. This notebook teaches you how to build production-grade structured outputs that actually hold their shape.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Why plain prompting isn't enough |
| 2 | **Setup** — Install + API key |
| 3 | **Pydantic Schema** — `StructuredResponse` with field validators |
| 4 | **Guard Construction** — `create_guard()` and the system prompt |
| 5 | **Validation Loop** — `validate_response()` and the reask mechanism |
| 6 | **Running Test Cases** — 4 prompts: valid, too-long, no-JSON, URL |
| 7 | **Audit Trail** — Reading `guard.history` |
| 8 | **Failure Modes** — When reasks are exhausted |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier works fine)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `guardrails-ai` Python package (installed in Cell 2)

### Key References
> Guardrails AI Documentation — https://www.guardrailsai.com/docs  
> Pydantic v2 Validators — https://docs.pydantic.dev/latest/concepts/validators/  
> "Reliable, Adaptable, and Attributable Language Models with Retrieval" (inspiration for structured output pipelines) — https://arxiv.org/abs/2307.11003

In [ ]:
# Cell 2 — Install dependencies (Colab auto-detects; local skips)
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "guardrails-ai", "pydantic", "openai", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install. Run: pip install guardrails-ai")

In [ ]:
# Cell 3 — API key setup (Colab Secrets or local .env)
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not key.startswith('sk-'):
    raise RuntimeError("OPENAI_API_KEY must be set before running this live workbook.")
print("API key ready.")

---
## Part 1 — Why Plain Prompting Isn't Enough

### The structured output problem

When you ask an LLM to respond in JSON, several things can go wrong:

| Failure mode | Example |
|---|---|
| Missing fields | `{"summary": "..."}` — no `points`, no `safe` |
| Wrong types | `"safe": "yes"` instead of `"safe": true` |
| Constraint violation | summary is 350 chars, not 200 |
| Contamination | A URL slips into `points` despite the instruction |
| Markdown wrapping | ` ```json\n{...}\n``` ` — JSON buried in markdown |

### The naive fix — just retry

Many teams implement a retry loop: if JSON parsing fails, call the model again. But:
- **Blind retries** don't tell the model *what* was wrong
- **No feedback loop** — same prompt → same probability of the same mistake
- **No audit** — you can't tell how many retries a response took in production

### The Guardrails approach — reask with correction

```
User prompt
    │
    ▼
┌─────────────────────────────────────────────────────┐
│                    Guard.__call__()                 │
│                                                     │
│  attempt 1: LLM call → parse → validate            │
│    ✓ passes? → return result                        │
│    ✗ fails?  → build reask message with errors      │
│                                                     │
│  attempt 2: LLM call (with correction) → validate  │
│    ✓ passes? → return result                        │
│    ✗ fails?  → build reask message again            │
│                                                     │
│  attempt N (max_reasks): final try or raise         │
└─────────────────────────────────────────────────────┘
```

Each reask message includes the *specific* validation error — e.g., "summary must be <= 200 chars, got 312". The model now has actionable feedback.

Guardrails AI also keeps a full **history** of every attempt so you can audit, debug, and measure reask rates in production.

---
## Part 2 — The Pydantic Schema (`src/tools.py`)

Guardrails AI uses **Pydantic v2** as its validation layer. This means your output schema is just a normal Pydantic model — with the addition of `@field_validator` methods that encode the business rules the LLM must obey.

### Our schema: `StructuredResponse`

```
StructuredResponse
├── summary: str         ← max 200 characters
├── points: list[str]    ← exactly 3 items, no URLs
└── safe: bool           ← content safety flag
```

### Why field validators?

Pydantic's built-in type system handles *type* coercion (`"true"` → `True`, `"42"` → `42`). But **business rules** need custom validators:

- `summary_length`: checks `len(v) <= 200` — not expressible in JSON Schema alone
- `no_urls_in_points`: uses a regex to scan each point for `http://` or `https://`

When a validator raises `ValueError`, Guardrails catches it and adds the error message to the reask prompt. This is the key mechanism that makes reasks *informative* instead of blind.

In [ ]:
# Part 2 — Define the StructuredResponse Pydantic model
# (This is the full content of src/tools.py)

from pydantic import BaseModel, Field, field_validator
import re

class StructuredResponse(BaseModel):
    """Schema the LLM must conform to. Guardrails triggers reask if violated."""
    summary: str = Field(description="A brief summary, max 200 chars")
    points: list[str] = Field(description="Exactly 3 bullet points")
    safe: bool = Field(description="Is the content safe and appropriate?")

    @field_validator("summary")
    @classmethod
    def summary_length(cls, v: str) -> str:
        if len(v) > 200:
            raise ValueError(f"summary must be <= 200 chars, got {len(v)}")
        return v

    @field_validator("points")
    @classmethod
    def no_urls_in_points(cls, v: list[str]) -> list[str]:
        if len(v) != 3:
            raise ValueError(f"points must contain exactly 3 items, got {len(v)}")
        url_pattern = re.compile(r"https?://\S+")
        for point in v:
            if url_pattern.search(point):
                raise ValueError("points must not contain URLs")
        return v

print("StructuredResponse schema defined.")
print("Fields:", list(StructuredResponse.model_fields.keys()))

In [ ]:
# Part 2b — Test the validators directly (no LLM needed)
from pydantic import ValidationError

# Valid input — should pass
try:
    valid = StructuredResponse(
        summary="Water is essential for life.",
        points=["Hydration improves focus", "Regulates body temperature", "Aids digestion"],
        safe=True
    )
    print("VALID   :", valid.summary)
except ValidationError as e:
    print("FAILED  :", e)

# Too-long summary — should fail
try:
    bad_summary = StructuredResponse(
        summary="A" * 250,  # 250 chars, exceeds 200
        points=["Point 1", "Point 2", "Point 3"],
        safe=True
    )
except ValidationError as e:
    print("SUMMARY ERR:", e.errors()[0]["msg"])

# URL in points — should fail
try:
    bad_url = StructuredResponse(
        summary="Short summary",
        points=["Visit https://example.com", "Point 2", "Point 3"],
        safe=True
    )
except ValidationError as e:
    print("URL ERR    :", e.errors()[0]["msg"])

# Wrong point count — should fail (Pydantic min_length/max_length)
try:
    bad_count = StructuredResponse(
        summary="Short",
        points=["Only one point"],
        safe=True
    )
except ValidationError as e:
    print("COUNT ERR  :", e.errors()[0]["msg"])

---
## Part 3 — Building the Guard (`src/workflow.py`)

The `Guard` is Guardrails AI's core object. Think of it as a **middleware layer** that sits between your code and the LLM:

```
Your code  →  Guard.__call__()  →  OpenAI API  →  Guard validates  →  Your code
```

### `Guard.for_pydantic()`

This factory method wires a Pydantic model into the Guard:

```python
guard = Guard.for_pydantic(
    output_class=StructuredResponse,   # the schema to validate against
    messages=[...],                    # system prompt baked into every call
    num_reasks=2,                      # max correction attempts
)
```

### The system prompt strategy

The system prompt is **also** part of the enforcement strategy. Before Guardrails even runs validation, you want the model to *try* to produce valid output. A clear, schema-aligned system prompt reduces reask frequency:

- Specify the JSON shape explicitly
- State constraints directly: "max 200 chars", "exactly 3 points", "no URLs"
- Use `"Respond ONLY in valid JSON"` to prevent markdown wrapping

The Guard validation is the **fallback** for when the model still drifts — not the primary enforcement mechanism.

In [ ]:
# Part 3 — System prompt and Guard construction
# (This is the full content of src/workflow.py)

from guardrails import Guard

SYSTEM_PROMPT = """You are a concise AI assistant. Respond ONLY in valid JSON matching this schema:
{
  "summary": "<brief summary, max 200 chars>",
  "points": ["<point 1>", "<point 2>", "<point 3>"],
  "safe": true
}
Provide exactly 3 points. No URLs. No prose outside the JSON."""
REASK_LIMITS: dict[int, int] = {}


def create_guard(max_reasks: int = 2) -> Guard:
    guard = Guard.for_pydantic(
        output_class=StructuredResponse,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}],
    )
    guard.configure(num_reasks=max_reasks, allow_metrics_collection=False)
    REASK_LIMITS[id(guard)] = max_reasks
    return guard


guard = create_guard(max_reasks=2)
print("Guard created.")
print(f"  Schema : StructuredResponse")
print(f"  Reasks : up to 2")
print(f"  Model  : gpt-5.4-nano (set in __call__)")

---
## Part 4 — The Validation + Reask Loop

### How `guard()` works internally

```
guard(
    model="gpt-4o-mini",
    msg_history=[{"role": "user", "content": prompt}]
)
```

Internally, Guardrails:

1. **Calls the LLM** with the system prompt + user message
2. **Parses** the response (strips markdown, extracts JSON)
3. **Validates** with Pydantic — runs all `@field_validator` methods
4. If valid → returns `ValidationOutcome` with `validated_output`
5. If invalid → builds a correction prompt:  
   *"Your previous response had these errors: [summary must be <= 200 chars, got 312]. Please try again."*
6. **Reasks** the LLM with the correction prompt
7. Repeats up to `num_reasks` times
8. If still failing → raises `ValidationError` or returns the best partial output

### The `ValidationOutcome` object

```
result.validated_output   # dict or None
result.reasks             # list of reask attempts
result.raw_llm_output     # the raw string from the last LLM call
```

### Our `validate_response()` wrapper

We wrap the guard call in a try/except to return a consistent dict — useful for batch processing where you want to collect both passing and failing results without crashing.

In [ ]:
# Part 4 — The validate_response() wrapper function

def reask_limit(guard: Guard) -> int:
    return REASK_LIMITS.get(id(guard), getattr(guard, '_num_reasks', 2) or 0)


def validate_response(guard: Guard, prompt: str) -> dict:
    try:
        result = guard(
            model="gpt-5.4-nano",
            messages=[{"role": "user", "content": prompt}],
            num_reasks=reask_limit(guard),
        )
        return {
            "validated": result.validated_output,
            "reasks": [None] * max(len(getattr(guard.history.last, 'iterations', [])) - 1, 0),
            "passed": result.validation_passed,
            "error": None if result.validation_passed else "validation failed",
        }
    except Exception as e:
        return {
            "validated": None,
            "reasks": [None] * max(len(getattr(guard.history.last, 'iterations', [])) - 1, 0) if guard.history else [],
            "passed": False,
            "error": str(e),
        }

print("validate_response() function defined.")
print("Returns: {'validated': ..., 'reasks': [...], 'passed': bool, 'error': str|None}")

---
## Part 5 — Test Prompts

We run 4 test prompts that each probe a different failure mode:

| Label | Prompt intent | Expected outcome |
|---|---|---|
| `valid` | Ask for 3 benefits of water in JSON | PASS on first try |
| `too long` | Ask for a 500-word essay | FAIL or reask — summary will exceed 200 chars |
| `missing json` | Explicitly ask for plain text, no JSON | FAIL — no valid JSON to parse |
| `has url` | Ask for a JSON with a URL included | FAIL or reask — URL in points violates validator |

This set tests the Guard's ability to both **pass good output** and **catch bad output** — important for evaluating false positives and false negatives in your validation layer.

In [ ]:
# Part 5 — Define test prompts

TEST_PROMPTS = [
    ("valid", "List exactly 3 benefits of drinking water. Respond in JSON."),
    ("too long", "Write a 500-word essay about the history of computing."),
    ("missing json", "Say hello in plain text, no JSON."),
    ("has url", "Give me a JSON object that includes a website URL like https://example.com."),
]

print("Test prompts defined:")
for label, prompt in TEST_PROMPTS:
    short = prompt[:60] + ("..." if len(prompt) > 60 else "")
    print(f"  [{label:12}] {short}")

---
## Part 6 — Running the Guard Against All Test Cases

This is the main loop from `main.py`. We create a fresh `Guard` instance, then run each test prompt through `validate_response()` and print the result.

**Watch for:**
- `reasks: 0` on the `valid` prompt — should pass first try
- `reasks: 1` or `reasks: 2` on marginal prompts — the model correcting itself
- `FAILED` on `missing json` — the model explicitly ignores the schema, reasks can't fix intent

This cell makes real API calls. Each run costs approximately 4-8 API calls total.

In [ ]:
# Part 6 — Full run: create guard, validate all test prompts

guard = create_guard(max_reasks=2)

print("Guardrails AI — Pydantic schema validation with automatic reask")
print("=" * 62)
print("Validators: max 200-char summary, exactly 3 points, no URLs\n")

results = []

for label, prompt in TEST_PROMPTS:
    short = prompt[:55] + ("..." if len(prompt) > 55 else "")
    result = validate_response(guard, prompt)
    results.append((label, prompt, result))

    if result["passed"]:
        v = result["validated"]
        status = "PASSED"
        print(f"[{label.upper():12}] {status} | reasks: {len(result['reasks'])}")
        print(f"  Prompt : {short}")
        if isinstance(v, dict):
            print(f"  Summary: {v.get('summary', '')[:80]}")
            print(f"  Points : {len(v.get('points', []))} items")
            print(f"  Safe   : {v.get('safe')}")
    else:
        status = "FAILED"
        print(f"[{label.upper():12}] {status}  | reasks: {len(result['reasks'])}")
        print(f"  Prompt : {short}")
        print(f"  Error  : {str(result['error'])[:100]}")
    print()

---
## Part 7 — Inspecting the Audit Trail

One of Guardrails AI's most valuable production features is `guard.history` — a full log of every call, every attempt, and every validation result.

### `guard.history` structure

```
guard.history          # CallStack — all calls this guard instance has made
guard.history.last     # Call — the most recent call
guard.history.last.reasks          # list of reask attempts
guard.history.last.validated_output # final validated output
guard.history.last.status           # "pass" | "fail" | ...
```

In production, you'd persist this to a database or logging service for:
- **Observability**: how often does each prompt type require reasks?
- **Debugging**: what was the exact validation error on call #4732?
- **Quality monitoring**: is the reask rate rising? (model drift indicator)

In [ ]:
# Part 7 — Inspect guard.history for the last call

print("=== Guard History Inspection ===")
print(f"Total calls logged: {len(guard.history)}\n")

if guard.history:
    last_call = guard.history.last
    print(f"Last call status     : {getattr(last_call, 'status', 'N/A')}")
    print(f"Reask count          : {max(len(getattr(last_call, 'iterations', [])) - 1, 0)}")
    
    validated = getattr(last_call, 'validated_output', None)
    if validated:
        print(f"Validated output     : {str(validated)[:100]}")
    else:
        print("Validated output     : None (call failed)")

print("\n=== Per-Call Summary ===")
for i, call in enumerate(guard.history):
    status = getattr(call, 'status', 'unknown')
    reask_count = max(len(getattr(call, 'iterations', [])) - 1, 0)
    print(f"Call {i+1}: status={status}, reasks={reask_count}")

---
## Part 8 — Understanding Failure Modes

### When does Guardrails give up?

After `num_reasks` failed attempts, Guardrails raises an exception (or returns a partial result depending on configuration). Understanding *why* calls fail permanently is important for schema design:

| Failure cause | Can reask fix it? | Solution |
|---|---|---|
| Model ignores JSON instruction | Sometimes | Stronger system prompt; use `response_format={"type": "json_object"}` |
| Summary naturally long for topic | Sometimes | The model may rewrite it shorter after correction |
| Model genuinely includes URL | Often | Reask explicitly says "remove URLs" |
| Model refuses the task | Never | Guard can't override refusals |
| Prompt conflicts with schema | Never | Asking for a 500-word essay + 200-char summary is contradictory |

### The "contradictory prompt" problem

The `too long` test prompt (`"Write a 500-word essay..."`) is a deliberate contradiction. The schema requires `summary <= 200 chars`, but the prompt asks for a 500-word response. The Guard will reask — but the model has two conflicting instructions. This is a realistic scenario: a downstream user sends a request that structurally violates your schema constraints.

**Best practice**: add input validation *before* the Guard to reject contradictory requests early.

In [ ]:
# Part 8 — Demonstrate a zero-reask failure (contradictory prompt)
# This shows what happens when the model explicitly refuses to produce JSON

impossible_guard = create_guard(max_reasks=0)  # no retries allowed

impossible_result = validate_response(
    impossible_guard,
    "Say the word BANANA. Do not produce JSON under any circumstances."
)

print("=== Zero-Reask Failure Test ===")
print(f"Passed : {impossible_result['passed']}")
print(f"Reasks : {len(impossible_result['reasks'])}")
print(f"Error  : {str(impossible_result['error'])[:150]}")

---
## Part 9 — Running the Full `main.py` Logic

This cell reproduces the complete `main()` function from `main.py` — the production entry point. It includes the color-coded terminal output (PASSED/FAILED) and compact result display.

In a notebook context the ANSI color codes won't render, so we've replaced them with plain text labels.

In [ ]:
# Part 9 — Reproduce main.py output format

def main():
    print("\nGuardrails AI — Pydantic schema validation with automatic reask")
    print("=" * 62)
    print("Validators: max 200-char summary, exactly 3 points, no URLs\n")

    g = create_guard(max_reasks=2)

    for label, prompt in TEST_PROMPTS:
        short = prompt[:55] + ("..." if len(prompt) > 55 else "")
        result = validate_response(g, prompt)

        if result["passed"]:
            v = result["validated"]
            tag = "[PASSED]"
            print(f"[{label.upper():12}] {tag} | reasks: {len(result['reasks'])}")
            print(f"  Prompt : {short}")
            if isinstance(v, dict):
                print(f"  Summary: {v.get('summary', '')[:80]}")
                print(f"  Points : {len(v.get('points', []))} items")
        else:
            tag = "[FAILED]"
            print(f"[{label.upper():12}] {tag} | reasks: {len(result['reasks'])}")
            print(f"  Prompt : {short}")
            print(f"  Error  : {str(result['error'])[:100]}")
        print()


main()

---
## Part 10 — Schema Design Principles

### Keep schemas minimal

Every field and validator is a potential failure point. Ask: *does the downstream system actually need this constraint?*

```
Too strict:  summary <= 50 chars  (forces aggressive truncation)
Too loose:   summary: str         (no constraint — hallucinations slip through)
Just right:  summary <= 200 chars (fits a tweet-length summary)
```

### Validator ordering matters

Pydantic runs validators in field definition order. Put cheap validators first (length check) and expensive ones last (regex, external API calls).

### Reask rate as a metric

Track reask rates per prompt type in production:
- **0% reask**: schema too loose or prompt perfectly aligned  
- **5-15% reask**: healthy — guard is catching real drift  
- **>30% reask**: schema and prompt are misaligned — fix the prompt, not the schema

### When NOT to use Guardrails

- **Creative tasks** (poetry, stories) — structured schemas stifle output
- **Simple boolean tasks** — overkill for yes/no responses
- **High-volume / low-latency** — each reask adds ~1-3s latency; budget accordingly

Use OpenAI's native `response_format={"type": "json_object"}` or `structured_outputs` for simple cases, and reserve Guardrails for schemas with complex business-rule validators.

In [ ]:
# Part 10 — Quick schema introspection helper

import json

def inspect_schema(model_class):
    """Print the JSON Schema that Guardrails sends to the LLM."""
    schema = model_class.model_json_schema()
    print(f"Schema for {model_class.__name__}:")
    print(json.dumps(schema, indent=2))
    print(f"\nValidators defined:")
    for name, validator in model_class.__validators__.items() if hasattr(model_class, '__validators__') else []:
        print(f"  @field_validator('{name}')")

inspect_schema(StructuredResponse)

---
## Part 11 — Batch Validation with Statistics

In production you'll run Guardrails over many requests. This cell adds a stats layer on top of the test run — tracking pass rate, average reasks, and failure reasons. This is the pattern you'd adapt for an eval harness or observability dashboard.

In [ ]:
# Part 11 — Batch validation with stats collection

from collections import defaultdict

def run_batch(test_prompts: list, max_reasks: int = 2) -> dict:
    """Run all test prompts and collect statistics."""
    g = create_guard(max_reasks=max_reasks)
    stats = {
        "total": len(test_prompts),
        "passed": 0,
        "failed": 0,
        "reask_counts": [],
        "failures": [],
    }

    for label, prompt in test_prompts:
        result = validate_response(g, prompt)
        reask_count = len(result["reasks"])
        stats["reask_counts"].append(reask_count)

        if result["passed"]:
            stats["passed"] += 1
        else:
            stats["failed"] += 1
            stats["failures"].append({
                "label": label,
                "error": str(result["error"])[:100],
                "reasks": reask_count,
            })

    stats["pass_rate"] = stats["passed"] / stats["total"] * 100
    stats["avg_reasks"] = sum(stats["reask_counts"]) / len(stats["reask_counts"])
    return stats


batch_stats = run_batch(TEST_PROMPTS, max_reasks=2)

print("=== Batch Validation Statistics ===")
print(f"Total prompts : {batch_stats['total']}")
print(f"Passed        : {batch_stats['passed']} ({batch_stats['pass_rate']:.0f}%)")
print(f"Failed        : {batch_stats['failed']}")
print(f"Avg reasks    : {batch_stats['avg_reasks']:.2f}")
print(f"\nFailure details:")
for f in batch_stats["failures"]:
    print(f"  [{f['label']}] reasks={f['reasks']} → {f['error']}")

---
## Part 12 — Extending the Schema

Real schemas are richer. Here we explore how to add new fields and validators without breaking the Guard integration. The key rule: **every new validator should raise `ValueError` with a human-readable message** — that message becomes the reask correction instruction.

We'll add:
- A `category` field constrained to an enum
- A `confidence` float in [0.0, 1.0]
- A `no_profanity` validator on the summary (simplified demo)

In [ ]:
# Part 12 — Extended schema with more validators

from pydantic import BaseModel, Field, field_validator
from typing import Literal
import re

class ExtendedResponse(BaseModel):
    """Extended schema with category, confidence, and profanity check."""
    summary: str = Field(description="A brief summary, max 200 chars")
    points: list[str] = Field(description="Exactly 3 bullet points")
    safe: bool = Field(description="Is the content safe and appropriate?")
    category: Literal["health", "technology", "science", "general"] = Field(
        description="Topic category"
    )
    confidence: float = Field(description="Confidence score 0.0-1.0", ge=0.0, le=1.0)

    @field_validator("summary")
    @classmethod
    def summary_length(cls, v: str) -> str:
        if len(v) > 200:
            raise ValueError(f"summary must be <= 200 chars, got {len(v)}")
        return v

    @field_validator("points")
    @classmethod
    def no_urls_in_points(cls, v: list[str]) -> list[str]:
        if len(v) != 3:
            raise ValueError(f"points must contain exactly 3 items, got {len(v)}")
        url_pattern = re.compile(r"https?://\S+")
        for point in v:
            if url_pattern.search(point):
                raise ValueError("points must not contain URLs")
        return v

    @field_validator("summary")
    @classmethod
    def no_profanity(cls, v: str) -> str:
        # Simplified profanity check for demo
        blocked = ["spam", "scam"]  # replace with real list
        for word in blocked:
            if word.lower() in v.lower():
                raise ValueError(f"summary contains blocked term: '{word}'")
        return v


print("ExtendedResponse schema:")
print(json.dumps(ExtendedResponse.model_json_schema(), indent=2))

In [ ]:
# Part 12b — Create a Guard for the extended schema and run one call

EXTENDED_SYSTEM_PROMPT = """You are a concise AI assistant. Respond ONLY in valid JSON:
{
  "summary": "<brief summary, max 200 chars>",
  "points": ["<point 1>", "<point 2>", "<point 3>"],
  "safe": true,
  "category": "<one of: health, technology, science, general>",
  "confidence": <float between 0.0 and 1.0>
}
Exactly 3 points. No URLs. No prose outside JSON."""

extended_guard = Guard.for_pydantic(
    output_class=ExtendedResponse,
    messages=[{"role": "system", "content": EXTENDED_SYSTEM_PROMPT}],
)
extended_guard.configure(num_reasks=2, allow_metrics_collection=False)

ext_result = validate_response(
    extended_guard,
    "What are the top 3 benefits of exercise for mental health?"
)

print(f"Passed : {ext_result['passed']}")
print(f"Reasks : {len(ext_result['reasks'])}")
if ext_result["passed"] and ext_result["validated"]:
    v = ext_result["validated"]
    print(f"Category   : {v.get('category')}")
    print(f"Confidence : {v.get('confidence')}")
    print(f"Summary    : {v.get('summary')}")
    print(f"Points     : {v.get('points')}")
else:
    print(f"Error : {ext_result['error']}")

---
## Exercises

### Exercise 1 — Add a new validator: minimum summary length

The current schema only enforces a *maximum* summary length (200 chars). Add a validator that also enforces a *minimum* of 20 characters.

Steps:
1. Add a `@field_validator("summary")` method named `summary_min_length`
2. Raise `ValueError` if `len(v) < 20`
3. Test it with `StructuredResponse(summary="Hi", points=[...], safe=True)` — should fail

### Exercise 2 — Track reask rate per label

Modify `run_batch()` to return a `reask_rate_by_label` dict, where each key is the prompt label and the value is `reask_count / max_reasks` (a float 0.0-1.0).

### Exercise 3 — Add a 4th test prompt

Add a new entry to `TEST_PROMPTS` that tests a case not yet covered: a prompt that produces exactly 2 bullet points (violating the `min_length=3` constraint). Observe whether the reask fixes it or not.

In [ ]:
# Exercise Answer Key

# --- Exercise 1: minimum summary length validator ---

from pydantic import BaseModel, Field, field_validator
import re

class StructuredResponseV2(BaseModel):
    """StructuredResponse with both min and max summary length."""
    summary: str = Field(description="A brief summary, 20-200 chars")
    points: list[str] = Field(description="Exactly 3 bullet points", min_length=3, max_length=3)
    safe: bool = Field(description="Is the content safe and appropriate?")

    @field_validator("summary")
    @classmethod
    def summary_min_length(cls, v: str) -> str:
        if len(v) < 20:
            raise ValueError(f"summary must be >= 20 chars, got {len(v)}")
        return v

    @field_validator("summary")
    @classmethod
    def summary_max_length(cls, v: str) -> str:
        if len(v) > 200:
            raise ValueError(f"summary must be <= 200 chars, got {len(v)}")
        return v

    @field_validator("points")
    @classmethod
    def no_urls_in_points(cls, v: list[str]) -> list[str]:
        url_pattern = re.compile(r"https?://\S+")
        for point in v:
            if url_pattern.search(point):
                raise ValueError("points must not contain URLs")
        return v


from pydantic import ValidationError

# Should fail — summary is too short
try:
    StructuredResponseV2(summary="Hi", points=["A", "B", "C"], safe=True)
except ValidationError as e:
    print("Ex1 CORRECT — short summary rejected:", e.errors()[0]["msg"])

# Should pass — summary is 25 chars
try:
    ok = StructuredResponseV2(
        summary="Exactly twenty chars min.",
        points=["A", "B", "C"],
        safe=True
    )
    print("Ex1 CORRECT — 25-char summary accepted:", ok.summary)
except ValidationError as e:
    print("Ex1 WRONG:", e)

print()

# --- Exercise 2: reask rate per label ---

def run_batch_with_rates(test_prompts: list, max_reasks: int = 2) -> dict:
    g = create_guard(max_reasks=max_reasks)
    reask_rate_by_label = {}

    for label, prompt in test_prompts:
        result = validate_response(g, prompt)
        reask_count = len(result["reasks"])
        reask_rate_by_label[label] = reask_count / max_reasks if max_reasks > 0 else 0.0

    return {"reask_rate_by_label": reask_rate_by_label}


# Note: Running this makes live API calls — comment out to save tokens
# rates = run_batch_with_rates(TEST_PROMPTS, max_reasks=2)
# print("Ex2 — Reask rates by label:")
# for label, rate in rates["reask_rate_by_label"].items():
#     print(f"  {label:12}: {rate:.2f} ({rate*100:.0f}% of max)")
print("Ex2 — run_batch_with_rates() defined (live call commented out to save tokens)")

print()

# --- Exercise 3: new test prompt for 2-point violation ---

TEST_PROMPTS_V2 = TEST_PROMPTS + [
    ("two points", "Give me exactly 2 reasons why sleep is important. Respond in JSON."),
]

print("Ex3 — Extended TEST_PROMPTS:")
for label, prompt in TEST_PROMPTS_V2:
    print(f"  [{label:12}] {prompt[:60]}")

print()
print("To run Ex3 with live API calls:")
print("  for label, prompt in TEST_PROMPTS_V2:")
print("      result = validate_response(guard, prompt)")
print("      print(label, result['passed'], len(result['reasks']))")

---
## Workshop Complete

### What you learned

- **Pydantic v2 validators** as the enforcement layer for LLM output constraints
- **`Guard.for_pydantic()`** as the middleware that wraps every LLM call
- **The reask loop** — how Guardrails automatically sends correction prompts with specific error messages
- **`guard.history`** as the audit trail for production observability
- **Failure modes** — when reasks can fix drift vs. when the prompt itself is contradictory
- **Schema design principles** — minimal, testable, validator-error messages as reask instructions

### Key takeaways

1. Your `@field_validator` error message **becomes** the reask correction instruction — write it for the model, not just for logs
2. Track reask rates in production — rising rates signal model drift or schema misalignment
3. Input validation before the Guard is complementary — reject contradictory requests before they consume API tokens
4. For simple type coercion, prefer OpenAI's native `structured_outputs`; use Guardrails for complex business-rule validators

### Next steps

**Next: Example 109** — explore how structured output constraints interact with multi-step agent workflows, where each node in the graph enforces its own schema.

---
*Workshop built from [examples/108-guardrails-ai](https://github.com/Esturban/agent/tree/master/examples/108-guardrails-ai)*